# AutoML (Automated Machine Learning)
AutoML refers to the automated process of selecting, configuring, and tuning machine learning models, eliminating the need for extensive manual intervention. It allows users, even without deep expertise, to train effective models by optimizing hyperparameters, engineering features, and selecting algorithms.

## FLAML

In [ ]:
import pandas as pd
import numpy as np
import random
import sys
import warnings

# NLP imports
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
import torch  # Kept only for BERT / feature extraction

# Scikit-Learn utilities
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, hamming_loss

# --- FLAML ---
from flaml import AutoML
from sklearn.multioutput import MultiOutputClassifier
from sklearn.base import clone

# Suppress excessive warnings
warnings.filterwarnings("ignore")

# --- 1. Initial Configuration ---
def set_seed(seed_value=42):
    np.random.seed(seed_value)
    random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_seed(42)

# Download NLTK resources silently
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpus/stopwords')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)

STOP_WORDS_PT = set(stopwords.words('portuguese'))
DOMAIN_STOP_WORDS = {
    'curso', 'aprendizagem', 'educação', 'gestão', 'avaliação', 'pessoas',
    'científica', 'inclusão', 'trabalho', 'ensino', 'servidores', 'uso',
    'objetivo', 'conhecimento', 'público', 'formação', 'conceitos'
}
ALL_STOP_WORDS = list(STOP_WORDS_PT.union(DOMAIN_STOP_WORDS))

# --- 2. Data Loading and Feature Functions (UNCHANGED LOGIC) ---

def _load_and_filter_data(csv_path, min_course_count=5):
    print("-> Loading and filtering data...")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        sys.exit("Error: CSV file not found.")
        
    df = df.dropna(subset=['courseName', 'comp_name', 'courseDescription'])
    
    df_agg = df.groupby('courseName', as_index=False).agg({
        'courseDescription': 'first',
        'comp_name': lambda x: list(set(x))
    })
    
    counts = df_agg['comp_name'].explode().value_counts()
    rare_comps = counts[counts < min_course_count].index
    
    df_agg['comp_name_filtered'] = df_agg['comp_name'].apply(
        lambda x: [c for c in x if c not in rare_comps]
    )
    df_filtered = df_agg[df_agg['comp_name_filtered'].apply(len) > 0].copy()
    df_filtered['combinedText'] = df_filtered['courseDescription'].fillna('')
    
    print(f"   Processed data: {len(df_filtered)} courses remaining.")
    return df_filtered

def _get_tfidf_features(texts):
    print("-> Generating TF-IDF features...")
    tfidf = TfidfVectorizer(
        max_features=2000, ngram_range=(1, 2), min_df=5, max_df=0.7,
        stop_words=ALL_STOP_WORDS, sublinear_tf=True
    )
    return tfidf.fit_transform(texts).toarray()

def _get_word2vec_features(texts, vector_size=300):
    print(f"-> Generating Word2Vec features (dim={vector_size})...")
    def clean_text(text):
        text = text.lower().translate(str.maketrans('', '', string.punctuation))
        return [w for w in word_tokenize(text) if w not in STOP_WORDS_PT and w.isalpha()]
    tokens = [clean_text(t) for t in texts]
    model = Word2Vec(
        sentences=tokens,
        vector_size=vector_size,
        window=5,
        min_count=2,
        workers=4
    )
    embeddings = []
    for t in tokens:
        valid = [model.wv[w] for w in t if w in model.wv]
        embeddings.append(np.mean(valid, axis=0) if valid else np.zeros(vector_size))
    return np.vstack(embeddings)

def _get_bert_embeddings(texts):
    print("-> Generating BERT features (this may take some time)...")
    model_name = 'neuralmind/bert-base-portuguese-cased'
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)
    batch_size = 32
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            truncation=True, 
            padding=True,
            max_length=128
        )
        with torch.no_grad():
            outputs = model(**inputs)
        # CLS token representation
        all_embeddings.append(outputs.last_hidden_state[:, 0, :].numpy())
    return np.vstack(all_embeddings)

def get_data_pipeline(embedding_type='tfidf', csv_path="dataset_ifrn_artigo.csv"):
    df = _load_and_filter_data(csv_path, min_course_count=5)
    texts = df['combinedText'].tolist()
    
    if embedding_type == 'tfidf':
        X = _get_tfidf_features(texts)
    elif embedding_type == 'word2vec':
        X = _get_word2vec_features(texts, vector_size=300)
    elif embedding_type == 'bert':
        X = _get_bert_embeddings(texts)
    else:
        raise ValueError("Embedding must be one of: 'tfidf', 'word2vec', or 'bert'")
    
    print("-> Binarizing labels...")
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(df['comp_name_filtered'])
    
    print("-> Splitting train and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    return X_train, X_test, y_train, y_test, mlb.classes_

# --- 3. Evaluation Function (UNCHANGED LOGIC, IMPROVED OUTPUT) ---

def evaluate_and_print(y_test, y_proba, mlb_classes):
    """Evaluates the model using full metrics for each Top-K setting."""
    print(f"\n{'='*60}\n DETAILED RESULTS BY TOP-K \n{'='*60}")
    total_samples = y_test.shape[0]
    k_values = [1, 3, 5, 7, 10]

    # If probabilities come as a list of arrays (common with MultiOutputClassifier)
    if isinstance(y_proba, list):
        # Expected: list of length = n_labels,
        # each item shaped (n_samples, n_classes_per_label)
        # We always take the probability of class "1"
        y_proba = np.column_stack([
            p[:, 1] if p.shape[1] > 1 else p[:, 0] for p in y_proba
        ])

    # If probabilities are 1D, Top-K evaluation is not meaningful
    if y_proba.ndim == 1:
        y_proba = y_proba.reshape(-1, 1)

    if y_proba.shape[1] != y_test.shape[1]:
        print(f"Warning: Probability shape {y_proba.shape} differs from test labels {y_test.shape}.")
        return

    for k in k_values:
        if k > y_proba.shape[1]:
            # Skip invalid Top-K values
            continue

        print(f"\n>>> TOP-{k} ANALYSIS (Forcing {k} predictions per course) <<<")
        y_pred_k = np.zeros_like(y_test)

        # Indices of the K highest probabilities for each sample
        top_k_indices = np.argsort(y_proba, axis=1)[:, -k:]
        
        for i in range(total_samples):
            y_pred_k[i, top_k_indices[i]] = 1
            
        f1_mic = f1_score(y_test, y_pred_k, average='micro')
        f1_mac = f1_score(y_test, y_pred_k, average='macro', zero_division=0)
        h_loss = hamming_loss(y_test, y_pred_k)
        
        total_hits = 0
        samples_with_hit = 0
        for i in range(total_samples):
            true_indices = np.where(y_test[i] == 1)[0]
            pred_indices = top_k_indices[i]
            hits = len(set(pred_indices) & set(true_indices))
            total_hits += hits
            if hits > 0:
                samples_with_hit += 1
        
        precision_at_k = total_hits / (total_samples * k)
        hit_rate_at_k = (samples_with_hit / total_samples) * 100
        
        print(f"{'-'*40}")
        print(f"Hamming Loss:        {h_loss:.4f}")
        print(f"F1 Score (Micro):    {f1_mic:.4f}")
        print(f"F1 Score (Macro):    {f1_mac:.4f}")
        print(f"{'-'*40}")
        print(f"Precision@{k}:        {precision_at_k:.4f}")
        print(f"Partial Hit@{k}:      {hit_rate_at_k:.2f}%")
        print(f"{'-'*40}")

# --- 4. FLAML Logic ---

def run_flaml(X_train, y_train, X_test, y_test, mlb_classes, time_limit=300):
    print(f"\n{'='*40}\nStarting FLAML (AutoML)\n{'='*40}")
    print(f"Total time budget: {time_limit} seconds.")
    print(f"Dimensions: X={X_train.shape}, y={y_train.shape} (Multi-label)\n")

    # FLAML does not directly support multi-label classification.
    # Strategy: use ONLY the first label column to identify a strong base estimator.
    y_train_single = y_train[:, 0]

    automl = AutoML()
    settings = {
        "time_budget": time_limit,
        "metric": "f1",          # Metric for the binary task (label column 0)
        "task": "classification",
        "log_file_name": "flaml_plaforedu.log",
        "seed": 42,
        # Optionally restrict estimators:
        # "estimator_list": ["lgbm", "xgboost", "rf", "extra_tree"],
    }

    print("Searching for the best base estimator with FLAML (label column 0)...")
    automl.fit(X_train=X_train, y_train=y_train_single, **settings)

    print("\nBest model found by FLAML:")
    print(automl.model)  # Internal pipeline / estimator

    # Clone the best estimator to use with MultiOutputClassifier
    base_estimator = clone(automl.model.estimator)

    print("\nTraining MultiOutputClassifier with the selected base estimator...")
    multi_clf = MultiOutputClassifier(base_estimator)
    multi_clf.fit(X_train, y_train)

    print("Generating probabilities for all competencies...")
    # Returns a list of arrays (n_samples, 2) per label
    prob_list = multi_clf.predict_proba(X_test)

    print("Evaluating multi-label results (Top-K)...")
    evaluate_and_print(y_test, prob_list, mlb_classes)

[nltk_data] Error loading punkt: HTTP Error 404: Not Found
[nltk_data] Error loading stopwords: HTTP Error 404: Not Found


In [ ]:
# You can control the embedding here: 'tfidf', 'word2vec', or 'bert'
embedding_type = "tfidf"
csv_path = "dataset_ifrn_artigo.csv"

X_train, X_test, y_train, y_test, classes_ = get_data_pipeline(
    embedding_type=embedding_type,
    csv_path=csv_path
)

run_flaml(X_train, y_train, X_test, y_test, classes_, time_limit=600)

-> Carregando e filtrando dados...
   Dados processados: 337 cursos restantes.
-> Gerando features TF-IDF...
-> Binarizando labels...
-> Dividindo treino e teste...

Iniciando FLAML (AutoML)
Tempo limite total: 600 segundos.
Dimensões: X=(269, 817), y=(269, 53) (Multi-label)

Buscando melhor estimador base com FLAML (coluna 0 de y)...
[flaml.automl.logger: 11-18 17:52:31] {1752} INFO - task = classification
[flaml.automl.logger: 11-18 17:52:31] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 11-18 17:52:31] {1862} INFO - Minimizing error metric: 1-f1
[flaml.automl.logger: 11-18 17:52:31] {1979} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'catboost', 'lrl1']
[flaml.automl.logger: 11-18 17:52:31] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 11-18 17:52:31] {2417} INFO - Estimated sufficient time budget=769s. Estimated necessary time budget=19s.
[flaml.automl.logger: 11-18 17:52:31] {2466} IN

In [ ]:
# You can control the embedding here: 'tfidf', 'word2vec', or 'bert'
embedding_type = "word2vec"
csv_path = "dataset_ifrn_artigo.csv"

X_train, X_test, y_train, y_test, classes_ = get_data_pipeline(
    embedding_type=embedding_type,
    csv_path=csv_path
)

run_flaml(X_train, y_train, X_test, y_test, classes_, time_limit=600)

-> Carregando e filtrando dados...
   Dados processados: 337 cursos restantes.
-> Gerando features Word2Vec (dim=300)...
-> Binarizando labels...
-> Dividindo treino e teste...

Iniciando FLAML (AutoML)
Tempo limite total: 600 segundos.
Dimensões: X=(269, 300), y=(269, 53) (Multi-label)

Buscando melhor estimador base com FLAML (coluna 0 de y)...
[flaml.automl.logger: 11-18 18:03:05] {1752} INFO - task = classification
[flaml.automl.logger: 11-18 18:03:05] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 11-18 18:03:05] {1862} INFO - Minimizing error metric: 1-f1
[flaml.automl.logger: 11-18 18:03:05] {1979} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'catboost', 'lrl1']
[flaml.automl.logger: 11-18 18:03:05] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 11-18 18:03:05] {2417} INFO - Estimated sufficient time budget=759s. Estimated necessary time budget=19s.
[flaml.automl.logger: 11-18 18:03:0

In [ ]:
# You can control the embedding here: 'tfidf', 'word2vec', or 'bert'
embedding_type = "bert"
csv_path = "dataset_ifrn_artigo.csv"

X_train, X_test, y_train, y_test, classes_ = get_data_pipeline(
    embedding_type=embedding_type,
    csv_path=csv_path
)

run_flaml(X_train, y_train, X_test, y_test, classes_, time_limit=600)

-> Carregando e filtrando dados...
   Dados processados: 337 cursos restantes.
-> Gerando features BERT (pode demorar)...
-> Binarizando labels...
-> Dividindo treino e teste...

Iniciando FLAML (AutoML)
Tempo limite total: 600 segundos.
Dimensões: X=(269, 768), y=(269, 53) (Multi-label)

Buscando melhor estimador base com FLAML (coluna 0 de y)...
[flaml.automl.logger: 11-18 18:19:05] {1752} INFO - task = classification
[flaml.automl.logger: 11-18 18:19:05] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 11-18 18:19:05] {1862} INFO - Minimizing error metric: 1-f1
[flaml.automl.logger: 11-18 18:19:05] {1979} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'catboost', 'lrl1']
[flaml.automl.logger: 11-18 18:19:05] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 11-18 18:19:06] {2417} INFO - Estimated sufficient time budget=1351s. Estimated necessary time budget=33s.
[flaml.automl.logger: 11-18 18:19

## AutoGluon

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import sys
import warnings

# NLP imports
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
import torch  # Kept only for BERT / feature extraction

# Scikit-Learn utilities
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss

# --- AutoGluon ---
from autogluon.tabular import TabularPredictor

# Suppress excessive warnings
warnings.filterwarnings("ignore")

# --- 1. Initial Configuration ---
def set_seed(seed_value=42):
    np.random.seed(seed_value)
    random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_seed(42)

# Download NLTK resources silently
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpus/stopwords')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)

STOP_WORDS_PT = set(stopwords.words('portuguese'))
DOMAIN_STOP_WORDS = {
    'curso', 'aprendizagem', 'educação', 'gestão', 'avaliação', 'pessoas',
    'científica', 'inclusão', 'trabalho', 'ensino', 'servidores', 'uso',
    'objetivo', 'conhecimento', 'público', 'formação', 'conceitos'
}
ALL_STOP_WORDS = list(STOP_WORDS_PT.union(DOMAIN_STOP_WORDS))

# --- 2. Data Loading and Feature Functions (LOGIC UNCHANGED) ---

def _load_and_filter_data(csv_path, min_course_count=5):
    print("-> Loading and filtering data...")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        sys.exit("Error: CSV file not found.")
        
    df = df.dropna(subset=['courseName', 'comp_name', 'courseDescription'])
    
    df_agg = df.groupby('courseName', as_index=False).agg({
        'courseDescription': 'first',
        'comp_name': lambda x: list(set(x))
    })
    
    counts = df_agg['comp_name'].explode().value_counts()
    rare_comps = counts[counts < min_course_count].index
    
    df_agg['comp_name_filtered'] = df_agg['comp_name'].apply(
        lambda x: [c for c in x if c not in rare_comps]
    )
    df_filtered = df_agg[df_agg['comp_name_filtered'].apply(len) > 0].copy()
    df_filtered['combinedText'] = df_filtered['courseDescription'].fillna('')
    
    print(f"   Processed data: {len(df_filtered)} courses remaining.")
    return df_filtered

def _get_tfidf_features(texts):
    print("-> Generating TF-IDF features...")
    tfidf = TfidfVectorizer(
        max_features=2000, ngram_range=(1, 2), min_df=5, max_df=0.7,
        stop_words=ALL_STOP_WORDS, sublinear_tf=True
    )
    return tfidf.fit_transform(texts).toarray()

def _get_word2vec_features(texts, vector_size=300):
    print(f"-> Generating Word2Vec features (dim={vector_size})...")
    def clean_text(text):
        text = text.lower().translate(str.maketrans('', '', string.punctuation))
        return [w for w in word_tokenize(text) if w not in STOP_WORDS_PT and w.isalpha()]
    tokens = [clean_text(t) for t in texts]
    model = Word2Vec(
        sentences=tokens,
        vector_size=vector_size,
        window=5,
        min_count=2,
        workers=4
    )
    embeddings = []
    for t in tokens:
        valid = [model.wv[w] for w in t if w in model.wv]
        embeddings.append(np.mean(valid, axis=0) if valid else np.zeros(vector_size))
    return np.vstack(embeddings)

def _get_bert_embeddings(texts):
    print("-> Generating BERT features (this may take some time)...")
    model_name = 'neuralmind/bert-base-portuguese-cased'
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)
    batch_size = 32
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            truncation=True, 
            padding=True,
            max_length=128
        )
        with torch.no_grad():
            outputs = model(**inputs)
        # CLS token representation
        all_embeddings.append(outputs.last_hidden_state[:, 0, :].numpy())
    return np.vstack(all_embeddings)

def get_data_pipeline(embedding_type='tfidf', csv_path="dataset_ifrn_artigo.csv"):
    df = _load_and_filter_data(csv_path, min_course_count=5)
    texts = df['combinedText'].tolist()
    
    if embedding_type == 'tfidf':
        X = _get_tfidf_features(texts)
    elif embedding_type == 'word2vec':
        X = _get_word2vec_features(texts, vector_size=300)
    elif embedding_type == 'bert':
        X = _get_bert_embeddings(texts)
    else:
        raise ValueError("Embedding must be one of: 'tfidf', 'word2vec', or 'bert'")
    
    print("-> Binarizing labels...")
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(df['comp_name_filtered'])
    
    print("-> Splitting train and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    return X_train, X_test, y_train, y_test, mlb.classes_

# --- 3. Evaluation Function (LOGIC UNCHANGED, ACCEPTS LIST INPUT) ---

def evaluate_and_print(y_test, y_proba, mlb_classes):
    """Evaluates the model using full metrics for each Top-K setting."""
    print(f"\n{'='*60}\n DETAILED RESULTS BY TOP-K \n{'='*60}")
    total_samples = y_test.shape[0]
    k_values = [1, 3, 5, 7, 10]

    # If probabilities come as a list of arrays (one per label)
    if isinstance(y_proba, list):
        # Expected: list of length = n_labels,
        # each item: (n_samples, n_classes)
        y_proba = np.column_stack([
            p[:, 1] if p.shape[1] > 1 else p[:, 0] for p in y_proba
        ])

    if y_proba.ndim == 1:
        y_proba = y_proba.reshape(-1, 1)

    if y_proba.shape[1] != y_test.shape[1]:
        print(f"Warning: Probability shape {y_proba.shape} differs from test labels {y_test.shape}.")
        return

    for k in k_values:
        if k > y_proba.shape[1]:
            continue

        print(f"\n>>> TOP-{k} ANALYSIS (Forcing {k} predictions per course) <<<")
        y_pred_k = np.zeros_like(y_test)

        top_k_indices = np.argsort(y_proba, axis=1)[:, -k:]
        
        for i in range(total_samples):
            y_pred_k[i, top_k_indices[i]] = 1
            
        f1_mic = f1_score(y_test, y_pred_k, average='micro')
        f1_mac = f1_score(y_test, y_pred_k, average='macro', zero_division=0)
        h_loss = hamming_loss(y_test, y_pred_k)
        
        total_hits = 0
        samples_with_hit = 0
        for i in range(total_samples):
            true_indices = np.where(y_test[i] == 1)[0]
            pred_indices = top_k_indices[i]
            hits = len(set(pred_indices) & set(true_indices))
            total_hits += hits
            if hits > 0:
                samples_with_hit += 1
        
        precision_at_k = total_hits / (total_samples * k)
        hit_rate_at_k = (samples_with_hit / total_samples) * 100
        
        print(f"{'-'*40}")
        print(f"Hamming Loss:        {h_loss:.4f}")
        print(f"F1 Score (Micro):    {f1_mic:.4f}")
        print(f"F1 Score (Macro):    {f1_mac:.4f}")
        print(f"{'-'*40}")
        print(f"Precision@{k}:        {precision_at_k:.4f}")
        print(f"Partial Hit@{k}:      {hit_rate_at_k:.2f}%")
        print(f"{'-'*40}")

# --- 4. MultilabelPredictor using multiple TabularPredictor instances ---

class MultilabelPredictor:
    """
    Simple wrapper for multi-label learning using multiple TabularPredictor models,
    one per label column.
    """
    def __init__(
        self,
        labels,
        path=None,
        problem_type="binary",
        eval_metric="f1",
        time_limit_per_label=60,
        presets="medium_quality_faster_train",
        seed=42,
    ):
        self.labels = list(labels)
        self.problem_type = problem_type
        self.eval_metric = eval_metric
        self.time_limit_per_label = time_limit_per_label
        self.presets = presets
        self.seed = seed
        self.path = path or "AutogluonModels_multilabel"
        os.makedirs(self.path, exist_ok=True)

        self.predictors = {}
        self.features = None  # set during fit

    def fit(self, train_data: pd.DataFrame):
        # Features = all columns except labels
        self.features = [c for c in train_data.columns if c not in self.labels]

        for label in self.labels:
            print(f"\nTraining TabularPredictor for label: {label}")
            label_path = os.path.join(self.path, f"label_{label}")
            os.makedirs(label_path, exist_ok=True)

            train_i = train_data[self.features + [label]]

            predictor = TabularPredictor(
                label=label,
                path=label_path,
                problem_type=self.problem_type,
                eval_metric=self.eval_metric,
            )

            predictor.fit(
                train_data=train_i,
                time_limit=self.time_limit_per_label,
                presets=self.presets,
                # Optional attempt to control randomness:
                # ag_args_fit={"random_state": self.seed},
            )

            self.predictors[label] = predictor

    def predict_proba(self, X: pd.DataFrame):
        """
        Returns a list of arrays (n_samples, n_classes) — one per label.
        """
        if self.features is not None and all(f in X.columns for f in self.features):
            X_feat = X[self.features]
        else:
            X_feat = X  # fallback

        proba_list = []
        for label in self.labels:
            predictor = self.predictors[label]
            # as_multiclass=True ensures 2 columns (class 0 and 1) for binary problems
            proba_df = predictor.predict_proba(X_feat, as_multiclass=True)
            proba_list.append(proba_df.values)

        return proba_list

# --- 5. AutoGluon logic ---

def run_autogluon(X_train, y_train, X_test, y_test, mlb_classes, time_limit=600):
    print(f"\n{'='*40}\nStarting AutoGluon (Tabular) for Multi-Label\n{'='*40}")
    print(f"Approximate total time budget: {time_limit} seconds.")
    print(f"Dimensions: X={X_train.shape}, y={y_train.shape} (Multi-label)\n")

    n_samples, n_features = X_train.shape
    n_labels = y_train.shape[1]

    # 1) Build train/test DataFrames (AutoGluon expects DataFrames)
    feature_cols = [f"f_{i}" for i in range(n_features)]
    label_cols = [f"label_{i}" for i in range(n_labels)]

    df_train = pd.DataFrame(X_train, columns=feature_cols)
    df_test = pd.DataFrame(X_test, columns=feature_cols)

    for i in range(n_labels):
        df_train[label_cols[i]] = y_train[:, i]
        df_test[label_cols[i]] = y_test[:, i]

    # 2) Split time budget per label (heuristic)
    time_per_label = max(30, time_limit // max(1, n_labels))
    print(f"Approximate time per label: {time_per_label} seconds (n_labels={n_labels})")

    ml_predictor = MultilabelPredictor(
        labels=label_cols,
        path="AutogluonModels_plaforedu",
        problem_type="binary",
        eval_metric="f1",
        time_limit_per_label=time_per_label,
        presets="medium_quality_faster_train",
        seed=42,
    )

    print("\nTraining models (one per competency)...")
    ml_predictor.fit(df_train)

    print("\nGenerating probabilities for all competencies...")
    # We only need features to predict
    df_test_features = df_test[feature_cols]
    prob_list = ml_predictor.predict_proba(df_test_features)

    print("\nEvaluating multi-label results (Top-K)...")
    evaluate_and_print(y_test, prob_list, mlb_classes)

In [ ]:
embedding_type = "tfidf"
csv_path = "dataset_ifrn_artigo.csv"

X_train, X_test, y_train, y_test, classes_ = get_data_pipeline(
    embedding_type=embedding_type,
    csv_path=csv_path
)

# Adjust the time_limit according to your machine and number of labels
run_autogluon(X_train, y_train, X_test, y_test, classes_, time_limit=1200)

Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.9
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Memory Avail:       11.46 GB / 15.58 GB (73.5%)
Disk Space Avail:   841.07 GB / 1006.85 GB (83.5%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'


-> Carregando e filtrando dados...
   Dados processados: 337 cursos restantes.
-> Gerando features TF-IDF...
-> Binarizando labels...
-> Dividindo treino e teste...

Iniciando AutoGluon (Tabular) para Multi-Label
Tempo limite total aproximado: 1200 segundos.
Dimensões: X=(269, 817), y=(269, 53) (Multi-label)

Tempo aproximado por label: 30 segundos (n_labels=53)

Treinando modelos (um por competência)...

Treinando TabularPredictor para label: label_0


Beginning AutoGluon training ... Time limit = 30s
AutoGluon will save models to "/home/morsinaldo/Desktop/dados/AutogluonModels_plaforedu/label_label_0"
Train Data Rows:    269
Train Data Columns: 817
Label Column:       label_0
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    11693.61 MB
	Train Data (Original)  Memory Usage: 1.68 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 3 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatur


Treinando TabularPredictor para label: label_1


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 3 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Original Features (Count: 57): ['f_1', 'f_3', 'f_5', 'f_28', 'f_29', 'f_127', 'f_149', 'f_156', 'f_225', 'f_243', 'f_298', 'f_322', 'f_329', 'f_333', 'f_337', 'f_338', 'f_345', 'f_352', 'f_363', 'f_400', 'f_413', 'f_414', 'f_415', 'f_452', 'f_475', 'f_480', 'f_487', 'f_515', 'f_523', 'f_524', 'f_533', 'f_545', 'f_546', 'f_561', 'f_564', 'f_575', 'f_583', 'f_593', 'f_594', 'f_595', 'f_597', 'f_602', 'f_610', 'f_615', 'f_616', 'f_627', 'f_632', 'f_650', 'f_674', 'f_691', 'f_700', 'f_709', 'f_712', 'f_725', 'f_740', 'f_747', 'f_775']
		These features were not used to generate any of the output features.

...


Treinando TabularPredictor para label: label_51


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 3 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Original Features (Count: 57): ['f_1', 'f_3', 'f_5', 'f_28', 'f_29', 'f_127', 'f_149', 'f_156', 'f_225', 'f_243', 'f_298', 'f_322', 'f_329', 'f_333', 'f_337', 'f_338', 'f_345', 'f_352', 'f_363', 'f_400', 'f_413', 'f_414', 'f_415', 'f_452', 'f_475', 'f_480', 'f_487', 'f_515', 'f_523', 'f_524', 'f_533', 'f_545', 'f_546', 'f_561', 'f_564', 'f_575', 'f_583', 'f_593', 'f_594', 'f_595', 'f_597', 'f_602', 'f_610', 'f_615', 'f_616', 'f_627', 'f_632', 'f_650', 'f_674', 'f_691', 'f_700', 'f_709', 'f_712', 'f_725', 'f_740', 'f_747', 'f_775']
		These features were not used to generate any of the output features.


Treinando TabularPredictor para label: label_52


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 3 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Original Features (Count: 57): ['f_1', 'f_3', 'f_5', 'f_28', 'f_29', 'f_127', 'f_149', 'f_156', 'f_225', 'f_243', 'f_298', 'f_322', 'f_329', 'f_333', 'f_337', 'f_338', 'f_345', 'f_352', 'f_363', 'f_400', 'f_413', 'f_414', 'f_415', 'f_452', 'f_475', 'f_480', 'f_487', 'f_515', 'f_523', 'f_524', 'f_533', 'f_545', 'f_546', 'f_561', 'f_564', 'f_575', 'f_583', 'f_593', 'f_594', 'f_595', 'f_597', 'f_602', 'f_610', 'f_615', 'f_616', 'f_627', 'f_632', 'f_650', 'f_674', 'f_691', 'f_700', 'f_709', 'f_712', 'f_725', 'f_740', 'f_747', 'f_775']
		These features were not used to generate any of the output features.


Gerando probabilidades em todas as competências...

Avaliando resultados multi-label (Top-K)...

 RESULTADOS DETALHADOS POR TOP-K 

>>> ANÁLISE TOP-1 (Forçando 1 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.0594
F1 Score (Micro):  0.1894
F1 Score (Macro):  0.0415
----------------------------------------
Precision@1:       0.3676
Acerto Parcial@1:  36.76%
----------------------------------------

>>> ANÁLISE TOP-3 (Forçando 3 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.0849
F1 Score (Micro):  0.2350
F1 Score (Macro):  0.0479
----------------------------------------
Precision@3:       0.2304
Acerto Parcial@3:  48.53%
----------------------------------------

>>> ANÁLISE TOP-5 (Forçando 5 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.1154
F1 Score (Micro):  0.2239
F1 Score (Macro):  0.0591
----------------------------------------
Precision@5:       0.1765
Acerto P

In [ ]:
embedding_type = "word2vec"
csv_path = "dataset_ifrn_artigo.csv"

X_train, X_test, y_train, y_test, classes_ = get_data_pipeline(
    embedding_type=embedding_type,
    csv_path=csv_path
)

# Adjust the time_limit according to your machine and number of labels
run_autogluon(X_train, y_train, X_test, y_test, classes_, time_limit=1200)

-> Carregando e filtrando dados...
   Dados processados: 337 cursos restantes.
-> Gerando features Word2Vec (dim=300)...


Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.9
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Memory Avail:       10.51 GB / 15.58 GB (67.5%)
Disk Space Avail:   840.76 GB / 1006.85 GB (83.5%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 30s
AutoGluon will save models to "/home/morsinaldo/Desktop/dados/AutogluonModels_plaforedu/label_label_0"
Train Data Rows:    269
Train Data Columns: 300
Label Column:       label_0
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerato

-> Binarizando labels...
-> Dividindo treino e teste...

Iniciando AutoGluon (Tabular) para Multi-Label
Tempo limite total aproximado: 1200 segundos.
Dimensões: X=(269, 300), y=(269, 53) (Multi-label)

Tempo aproximado por label: 30 segundos (n_labels=53)

Treinando modelos (um por competência)...

Treinando TabularPredictor para label: label_0


	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 300 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 300 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	0.3s = Fit runtime
	300 features in original data used to generate 300 features in processed data.
	Train Data (Processed) Memory Usage: 0.31 MB (0.0% of available memory)
Data preprocessing and feature engineering runtime = 0.3s ...
AutoGluon will gauge predictive performance using evaluation metric: 'f1'
	To change this, specify the eval_metric parameter of Predictor()
Automatically generating train/validation split with holdout_frac=0.2, Train Rows: 215, Val Rows: 54
User-specified model 


Treinando TabularPredictor para label: label_1


...


Treinando TabularPredictor para label: label_51


		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 300 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 300 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	0.3s = Fit runtime
	300 features in original data used to generate 300 features in processed data.
	Train Data (Processed) Memory Usage: 0.31 MB (0.0% of available memory)
Data preprocessing and feature engineering runtime = 0.3s ...
AutoGluon will gauge predictive performance using evaluation metric: 'f1'
	To change this, specify the eval_metric parameter of Predictor()
Automatically generating train/validation split with holdout_frac=0.2, Train Rows: 215, Val Rows: 54
User-specified model hyperparameters to be fit:
{
	'NN_TORCH': [{}],
	'GBM': [{'extra_trees': True,


Treinando TabularPredictor para label: label_52


		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 300 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 300 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	0.2s = Fit runtime
	300 features in original data used to generate 300 features in processed data.
	Train Data (Processed) Memory Usage: 0.31 MB (0.0% of available memory)
Data preprocessing and feature engineering runtime = 0.26s ...
AutoGluon will gauge predictive performance using evaluation metric: 'f1'
	To change this, specify the eval_metric parameter of Predictor()
Automatically generating train/validation split with holdout_frac=0.2, Train Rows: 215, Val Rows: 54
User-specified model hyperparameters to be fit:
{
	'NN_TORCH': [{}],
	'GBM': [{'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}}, {}, {'learning_rate': 


Gerando probabilidades em todas as competências...

Avaliando resultados multi-label (Top-K)...

 RESULTADOS DETALHADOS POR TOP-K 

>>> ANÁLISE TOP-1 (Forçando 1 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.0660
F1 Score (Micro):  0.0985
F1 Score (Macro):  0.0160
----------------------------------------
Precision@1:       0.1912
Acerto Parcial@1:  19.12%
----------------------------------------

>>> ANÁLISE TOP-3 (Forçando 3 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.1021
F1 Score (Micro):  0.0800
F1 Score (Macro):  0.0198
----------------------------------------
Precision@3:       0.0784
Acerto Parcial@3:  23.53%
----------------------------------------

>>> ANÁLISE TOP-5 (Forçando 5 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.1365
F1 Score (Micro):  0.0821
F1 Score (Macro):  0.0202
----------------------------------------
Precision@5:       0.0647
Acerto P

In [ ]:
embedding_type = "bert"
csv_path = "dataset_ifrn_artigo.csv"

X_train, X_test, y_train, y_test, classes_ = get_data_pipeline(
    embedding_type=embedding_type,
    csv_path=csv_path
)

# Adjust the time_limit according to your machine and number of labels
run_autogluon(X_train, y_train, X_test, y_test, classes_, time_limit=1200)

-> Carregando e filtrando dados...
   Dados processados: 337 cursos restantes.
-> Gerando features BERT (pode demorar)...


Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.9
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Memory Avail:       9.82 GB / 15.58 GB (63.0%)
Disk Space Avail:   840.86 GB / 1006.85 GB (83.5%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 30s
AutoGluon will save models to "/home/morsinaldo/Desktop/dados/AutogluonModels_plaforedu/label_label_0"
Train Data Rows:    269
Train Data Columns: 768
Label Column:       label_0
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator

-> Binarizando labels...
-> Dividindo treino e teste...

Iniciando AutoGluon (Tabular) para Multi-Label
Tempo limite total aproximado: 1200 segundos.
Dimensões: X=(269, 768), y=(269, 53) (Multi-label)

Tempo aproximado por label: 30 segundos (n_labels=53)

Treinando modelos (um por competência)...

Treinando TabularPredictor para label: label_0


	Available Memory:                    10059.71 MB
	Train Data (Original)  Memory Usage: 0.79 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 768 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 768 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	0.5s = Fit runtime
	768 features in original data used to generate 768 features in processed data.
	Train Data (Processed) Memory Usage: 0.79 MB (0.0% of available memory)
Data pr


Treinando TabularPredictor para label: label_1


...


Treinando TabularPredictor para label: label_51


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 768 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 768 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	0.4s = Fit runtime
	768 features in original data used to generate 768 features in processed data.
	Train Data (Processed) Memory Usage: 0.79 MB (0.0% of available memory)
Data preprocessing and feature engineering runtime = 0.45s ...
AutoGluon will gauge predictive performance using evaluation metric: 'f1'
	To change this, specify the eval_metric parameter of Predictor()
Automatically generating train/validation split with holdout_fra


Treinando TabularPredictor para label: label_52


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 768 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 768 | ['f_0', 'f_1', 'f_2', 'f_3', 'f_4', ...]
	0.4s = Fit runtime
	768 features in original data used to generate 768 features in processed data.
	Train Data (Processed) Memory Usage: 0.79 MB (0.0% of available memory)
Data preprocessing and feature engineering runtime = 0.45s ...
AutoGluon will gauge predictive performance using evaluation metric: 'f1'
	To change this, specify the eval_metric parameter of Predictor()
Automatically generating train/validation split with holdout_fra


Gerando probabilidades em todas as competências...

Avaliando resultados multi-label (Top-K)...

 RESULTADOS DETALHADOS POR TOP-K 

>>> ANÁLISE TOP-1 (Forçando 1 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.0538
F1 Score (Micro):  0.2652
F1 Score (Macro):  0.0704
----------------------------------------
Precision@1:       0.5147
Acerto Parcial@1:  51.47%
----------------------------------------

>>> ANÁLISE TOP-3 (Forçando 3 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.0766
F1 Score (Micro):  0.3100
F1 Score (Macro):  0.0938
----------------------------------------
Precision@3:       0.3039
Acerto Parcial@3:  63.24%
----------------------------------------

>>> ANÁLISE TOP-5 (Forçando 5 previsões por curso) <<<
----------------------------------------
Hamming Loss:      0.1060
F1 Score (Micro):  0.2873
F1 Score (Macro):  0.1212
----------------------------------------
Precision@5:       0.2265
Acerto P